In [38]:
# transfromer 하이퍼파라미터 및 각종 환경 설정
import torch
import torch.nn as nn 
import torch.optim as optim
import torch.nn.functional as F 
import math

d_model = 512  # 모델 임베딩 차원
num_layers = 6  # Encoder / Decoder 블록(레이어) 수
num_heads = 8  # Multi-Head Attention 헤드 수
d_k = d_model // num_heads  # 헤드당 Q, K, V 차원 수 (512//8 = 64)
d_ff = 2048  # Position-wise FFN의 은닉 차원
drop_out_rate = 0.1  # Dropout 비율(과적합 완화)

src_vocab_size = 10000  # 원문 사전 크기
trg_vocab_size = 10000  # 번역문 사전 크기

batch_size = 64
seq_len = 128  # 최대 시퀀스 길이 (패딩 처리)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [39]:
# Transformer 모델 뼈대 정의 (Embedding -> Encoder -> Decoder -> Output)

# 토큰 임베딩에 위치 정보(순서)를 더해주는 Positional Encoding 모듈
class PositionalEncoding(nn.Module):
    pass

# 소스 입력을 인코딩해 문맥 표현을 만드는 Enoder
class Encoder(nn.Module):
    pass

# 타겟 입력과 encoder outputs를 이용해 디코딩 출력 시퀀스 생성하는 Decoder
class Decoder(nn.Module):
    pass

In [40]:
# Transformer 전체 흐름 : 임베딩 -> 포지셔널 -> 인코더 -> 디코더 -> 출력층
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, d_model):
        super().__init__()  # 모듈 초기화
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)  # 소스(원문) 토큰 ID 받아서 ID를 임베딩(d_model) 작업
        self.trg_embedding = nn.Embedding(trg_vocab_size, d_model)  # 타겟(번역문) 토큰 ID 받아서 ID를 임베딩(d_model) 작업
        self.positional_encoding = PositionalEncoding()  # 위치 정보 인코딩 모듈
        self.encoder = Encoder()  # 인코더
        self.decoder = Decoder()  # 디코더
        self.output_layer = nn.Linear(d_model, trg_vocab_size)  # 디코더 출력(d_model) -> 어휘 로짓(vocab)
        self.softmax = nn.LogSoftmax(dim=1)  # 마지막 차원 기준 로그 확률 변환

    # 순전파 : src_inputs, trg_inputs를 받아서 번역문 토큰을 로그확률 분포로 출력
    def forward(self, src_inputs, trg_inputs, e_mask=None, d_mask=None):
        src_inputs = self.src_embedding(src_inputs)  # (B, T_src) -> (B, T_src, d_model)  # 차원추가
        src_inputs = self.positional_encoding(src_inputs)  # 위치 정보 추가

        trg_inputs = self.trg_embedding(trg_inputs)  # (B, T_src) -> (B, T_src, d_model)  # 차원추가
        trg_inputs = self.positional_encoding(trg_inputs)  # 위치 정보 추가

        encoder_outputs = self.encoder(src_inputs, e_mask)  # 소스(원문) 문장 인코딩

        decoder_outputs = self.decoder(trg_inputs, encoder_outputs, e_mask, d_mask)  # 디코딩 (인코더 마스크, 디코더 마스크가 있으면 적용) 결과 출력

        outputs = self.output_layer(decoder_outputs)  # (B, T_trg, d_model) -> (B, T_trg, trg_vocab_size)
        outputs = self.softmax(outputs)  # 각 시점별 타겟 토큰 로그확률 분포

        return outputs  # 최종 로그확률 반환

In [41]:
# PositionalEncoding 구현 (위치정보 추가) : 사인/코사인 함수로 위치정보 만들어 임베딩에 더해줌
class PositionalEncoding(nn.Module):
    def __init__(self, seq_len, d_model):
        super().__init__()

        pos_encoding = torch.zeros(seq_len, d_model)  # (T, E)로 위치 인코딩 행렬 생성

        for pos in range(seq_len):
            for i in range(d_model):
                if i % 2 == 0:  # 짝수 차원은 sin함수로 계산 (sin(pos/10000^(2i/d)))
                    pos_encoding[pos, i] = math.sin(pos / (10000 ** (2 * i / d_model)))
                else:  # 홀수 차원은 cos함수로 계산 (cos(pos/10000^(2i/d)))
                    pos_encoding[pos, i] = math.cos(pos / (10000 ** (2 * i / d_model)))

        pos_encoding = pos_encoding.unsqueeze(0)  # (seq_len, d_model) -> (1, seq_len, d_model)

        # pos_encoding은 학습 대상이 아님
        self.pos_encoding = pos_encoding.to(device).requires_grad_(False)

    def forward(self, x):
        x = x * math.sqrt(d_model)  # 임베딩 스케일을 늘려서 위치벡터와 규모를 맞춰줌
        x = x + self.pos_encoding  # (B, seq_len, d_model)에 위치 인코딩 더해줌
        return x  # 위치 정보가 반영된 임베딩 반환

In [42]:
# Transformer FFN(Position-wise Feed Forward) 레이어 구현 : 각 토큰 위치별로 동일한 2층 MLP 적용
class FeedForwardLayer(nn.Module):
    def __init__(self, d_model, d_ff, drop_out_rate):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)  # d_model -> d_ff 확장
        self.relu = nn.ReLU()  # 비선형 추가
        self.dropout = nn.Dropout(drop_out_rate)  # 과적합 완화
        self.linear2 = nn.Linear(d_ff, d_model)  # d_ff -> d_model 축소 (원래 차원으로 축소)

    def forward(self, x):
        x = self.linear1(x)  # (B, T, d_model) -> (B, T, d_ff)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)  # (B, T, d_ff) -> (B, T, d_model)
        return x  # FFN 결과 반환

In [43]:
# Transformer Layer Normalization 래퍼클래스 구현 : 입력의 마지막 차원(d_model)을 기준으로 정규화(Normalization)해서 학습을 한정화시키는 레이어
class LaterNormalization(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        # (마지막 차원) 정규화 모듈 생성
        # - elementwise_affine=True : 각 요소마다 다른 스케일/오프셋을 적용
        # - eps : 내부 분산 계산시 0으로 나누기가 되지 않도록 미세한 값
        self.layer = nn.LayerNorm([d_model], elementwise_affine=True, eps=eps)

    def forward(self, x):
        return self.layer(x)  # 정규화된 텐서 반환

In [44]:
# Multi-Head Attention : Q/K/V 선형변환 후 헤드로 분할하고, Scaled dot-product attention을 수행
class MultiheadAttention(nn.Module):
    def __init__(self, d_model, drop_out_rate):
        super().__init__()

        self.w_q = nn.Linear(d_model, d_model)  # Q 생성 선형층 (d_model -> d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(drop_out_rate)
        self.attn_softmax = nn.Softmax(dim=1)  # 마지막 축(T_k) 기준 softmax

        self.w_o = nn.Linear(d_model, d_model)  # 헤드 결합후에 사용하는 출력 선형층 (d_model -> d_model)

    # 헤드 분리하여 q, k, v 상태로 Scaled dot-product attention 계산하는 메서드
    def attention(self, q, k, v):
        attn_scores = torch.matmul(q, k.transpose(-1, -2))  # Q·K^T -> (B, h, T_q, T_k) 유사도
        attn_scores /= math.sqrt(d_k)  # 헤드차원 (d_k)로 나눠 softmax 수치 안정화

        attn_weights = self.attn_softmax(attn_scores)  # score -> attention 확률 분포

        attn_weights = self.dropout(attn_weights)  # attention_weight에 dropout 적용 (과적합 방지)

        output = torch.matmul(attn_weights, v)  # 확률 가중합으로 V 결합 -> (B, h, T_q, d_k)
        return output  # 헤드별 attention output 반환

    # 입력 q, k, v를 받아 multi_head attention 수행
    def forward(self, q, k, v, mask=None):
        B, T_q, _ = q.size()  # (B:batch, T_q=query 길이)
        _, T_k, _ = k.size()  # T_k = key 길이(k, v 길이 동일함)

        q = self.w_q(q)  # (B, T_q, d_model) -> (B, T_q, d_model)
        k = self.w_k(k)  # (B, T_k, d_model) -> (B, T_k, d_model)
        v = self.w_v(v)  # (B, T_k, d_model) -> (B, T_k, d_model)

        # 헤드 분할
        q_heads = q.view(B, T_q, num_heads, d_k).transpose(1, 2)   # (B, T_q, d_model) -> (B, num_head, T_q, d_k)
        k_heads = k.view(B, T_k, num_heads, d_k).transpose(1, 2)   # (B, T_k, d_model) -> (B, num_head, T_k, d_k)
        v_heads = v.view(B, T_k, num_heads, d_k).transpose(1, 2)   # (B, T_k, d_model) -> (B, num_head, T_k, d_k)

        # 어텐션 연산
        attn_value = self.attention(q_heads, k_heads, v_heads)   # (B, num_head, T_q, d_k) 헤드별 attention 결과

        # 헤드 결합
        output = attn_value.transpose(1, 2)  # (B, h, T_q, d_k) -> (B, T_q, h, d_k)
        output = output.contiguous().view(B, T_q, d_model)  # (B, T_q, n_h * d_k) -> (B, T_q, d_model)

        # 출력 선형변환
        output = self.w_o(output)  # 최종 출력 선형 변환 (d_model -> d_model)

        return output  # (B, T_q, d_model) 반환

B : batch_size   
n_h : 헤드 개수 (멀티헤드)    
T_q : Query의 시퀀스 길이(토큰 개수)    
T_k : 
d_k : 헤드당 차원수 (d_model/h)    
d_model = 전체 차원     

In [45]:
# Self-Attention + FFN을 Residual로 쌓은 Encoder 블록 
class EncoderLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm = LaterNormalization(d_model)  # Self-Attention 전 LayerNorm
        self.multihead_attention = MultiheadAttention(d_model, drop_out_rate) #  Multi-head Self-Attention
        self.drop_out = nn.Dropout(drop_out_rate)  # Residual에 적용할 Dropout

        self.layer_norm2 = LaterNormalization(d_model)  # FFN 전 LayerNorm
        self.feed_forward = FeedForwardLayer(d_model, d_ff, drop_out_rate)  # Position-wise FFN
        self.drop_out2 = nn.Dropout(drop_out_rate)  # Residual에 적용할 Dropout

    # 입력 x를 받아 (Self-Attention -> FFN) 두 서브레이어를 Residual 적용
    def forward(self, x, e_mask=None):
        x_1 = self.layer_norm(x)  # attention 전 정규화
        x = x + self.drop_out(  # Residual : x + Dropout(attn(x))
            self.multihead_attention(x_1, x_1, x_1, mask=e_mask)  # Q=K=V로 Self-Attention 수행
        )

        x_2 = self.layer_norm2(x)  # FFN 전 정규화
        x = x + self.drop_out2(  # Residual : x + Dropout(ffn(x))
            self.feed_forward(x_2)
        )
        return x  # (B, T, d_model) 반환

# EncoderLayer를 num_layers개 쌓고, 마지막에 LayerNorm을 적용하는 Encoder 스택 클래스
class Encoder(nn.Module):
    def __init__(self, d_model, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([EncoderLayer() for _ in range(num_layers)])  # EncoderLayer 반복 구성
        self.layer_norm = LaterNormalization(d_model)  # 최종 출력 정규화

    def forward(self, x, e_mask):
        for layer in self.layers:  # 레이어를 순차적으로 적용
            x = layer(x, e_mask)  # encoder 출력 + 마스크와 함께 전달하여 인코딩 수행

        return self.layer_norm(x)  # 최종 정규화하여 반환

In [46]:
# 디코더 : Masked Self-Attention -> Cross-Attention -> FFN로 Residual로 쌓은 디코더 블록
class DecoderLayer(nn.Module):
    def __init__(self, ):
        super().__init__()
        self.layer_norm1 = LaterNormalization(d_model)  # Masked Self-Attention 전 Layer-Norm
        # 미래 토큰 차단 self-attn
        self.masked_multihead_self_attention = MultiheadAttention(d_model, drop_out_rate)
        self.drop_out1 = nn.Dropout(drop_out_rate)  # Residual에 적용할 Dropout

        self.layer_norm2 = LaterNormalization(d_model)  # Cross-Attention 전에 적용할 Layer-Norm
        self.multihead_cross_attention = MultiheadAttention(d_model, drop_out_rate)  # encoder 출력과 Cross-Attention
        self.drop_out2 = nn.Dropout(drop_out_rate)  # Residual에 적용할 Dropout

        self.layer_norm3 = LaterNormalization(d_model)  # FFN 전 적용할 Layer-Norm
        self.feedforward = FeedForwardLayer(d_model, d_ff, drop_out_rate)  # Position-wise FFN
        self.drop_out3 = nn.Dropout(drop_out_rate)  # Residual에 적용할 Dropout

    # 입력 x(타겟 임베딩)과 encoder 출력(e_outputs)을 받아서 디코더 블록 출력 반환
    def forward(self, x, e_outputs, e_mask, d_mask):
        # Masked Self-Attention
        x_1 = self.layer_norm1(x)
        x = x + self.drop_out1(  # Residual : x + Dropout(masked_attn(x))
            self.masked_multihead_self_attention(x_1, x_1, x_1, mask=d_mask)  # Q=K=V, d_mask로 미래 토큰 차단
        )

        # Cross-Attention
        x_2 = self.layer_norm2(x)
        x = x + self.drop_out2(  # Residual : x + Dropout(cross_attn(x, enc))
            self.multihead_cross_attention(x_2, e_outputs, e_outputs, mask=e_mask)  # Q=디코더, K/V=인코더 출력
        )

        # feed_forward
        x_3 = self.layer_norm3(x)
        x = x + self.drop_out3(  # Residual : x + Dropout(ffn(x))
            self.feedforward(x_3)
        )
        return x  # (B, T_trg, d_model) 반환

########## ~깃허브~
class Decoder(nn.Module):
    def __init__(self, d_model, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([DecoderLayer() for _ in range(num_layers)])  # DecoderLayer 반복 구성
        self.layer_norm = LaterNormalization(d_model)  # 최종 출력 정규화

    def forward(self, x, e_outputs, e_mask, d_mask):
        for layer in self.layers:  # 레이어를 순차적으로 적용
            x = layer(x, e_outputs, e_mask, d_mask)  # encoder 출력 + 마스크와 함께 전달하여 인코딩 수행

        return self.layer_norm(x)  # 최종 정규화하여 반환

In [47]:
# Transformer 모델 : 소스/타겟 임베딩 + Positional Encoding + Encoder / Decoder + 출력층 
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, d_model, seq_len, num_layers):
        super().__init__()
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)  # 원문 토큰 ID -> 임베딩(d_model)
        self.trg_embedding = nn.Embedding(trg_vocab_size, d_model)  # 원문 토큰 ID -> 임베딩(d_model)
        self.positional_encoding = PositionalEncoding(seq_len, d_model)  # 위치 인코딩 추가
        self.encoder = Encoder(d_model, num_layers)  # 인코덛 스택
        self.decoder = Decoder(d_model, num_layers)  # 디코더 스택
        self.output_layer = nn.Linear(d_model, trg_vocab_size)  # 디코더 출력 -> 타겟 어휘 로짓
        self.softmax = nn.LogSoftmax(dim=1)  # 로짓 -> 로그 확률(마지막 차원 )           ######## ~깃허브~

    # src_inputs, trg_inputs를 받아 타겟 토큰 분포(로그확률)를 반환
    def forward(self, src_inputs, trg_inputs, e_mask=None, d_mask=None):
        src_inputs = self.src_embedding(src_inputs)  # (B, T_src) -> (B, T_src, d_model)
        src_inputs = self.positional_encoding(src_inputs)  # 위치 정보 추가

        trg_inputs = self.trg_embedding(trg_inputs)  # 여기 이하는 ~깃허브~ 주석 가져오기
        trg_inputs = self.positional_encoding(trg_inputs)

        encoder_outputs = self.encoder(src_inputs, e_mask)

        decoder_outputs = self.decoder(trg_inputs, encoder_outputs, e_mask, d_mask)

        outputs = self.output_layer(decoder_outputs)
        outputs = self.softmax(outputs)
        return outputs

model = Transformer(src_vocab_size, trg_vocab_size, d_model, seq_len, num_layers)
model

Transformer(
  (src_embedding): Embedding(10000, 512)
  (trg_embedding): Embedding(10000, 512)
  (positional_encoding): PositionalEncoding()
  (encoder): Encoder(
    (layers): ModuleList(
      (0-5): 6 x EncoderLayer(
        (layer_norm): LaterNormalization(
          (layer): LayerNorm((512,), eps=1e-06, elementwise_affine=True, bias=True)
        )
        (multihead_attention): MultiheadAttention(
          (w_q): Linear(in_features=512, out_features=512, bias=True)
          (w_k): Linear(in_features=512, out_features=512, bias=True)
          (w_v): Linear(in_features=512, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (attn_softmax): Softmax(dim=1)
          (w_o): Linear(in_features=512, out_features=512, bias=True)
        )
        (drop_out): Dropout(p=0.1, inplace=False)
        (layer_norm2): LaterNormalization(
          (layer): LayerNorm((512,), eps=1e-06, elementwise_affine=True, bias=True)
        )
        (feed_forward): 

In [ ]:
# 더미데이터 생성 후 forward 실행
src_inputs = torch.randint(0, src_vocab_size, (batch_size, seq_len)).to(device)  # (B, T) 더미데이터
trg_inputs = torch.randint(0, trg_vocab_size, (batch_size, seq_len)).to(device)  # (B, T) 더미데이터

model.eval()  # 평가모드
with torch.no_grad():
    output = model(src_inputs, trg_inputs)

print(f'src 입력 : {src_inputs.shape}')  # (B, T)
print(f'trg 입력 : {trg_inputs.shape}')  # (B, T)
print(f'최종 모델 출력 : {output.shape}')  # (B, T, trg_vocab_size)

src 입력 : torch.Size([64, 128])
trg 입력 : torch.Size([64, 128])
최종 모델 출력 : torch.Size([64, 128, 10000])


## Transformer 정리

Transformer는 Attention을 기반으로 문장의 토큰 간 관계를 학습하는 신경망 구조이다.

### Transformer의 주요 구성 요소

- **Embedding** : 토큰을 벡터로 변환
- **Positional Encoding** : 토큰의 위치와 순서 정보 추가
- **Self-Attention** : 문장 내 다른 토큰과의 관계 계산
- **Multi-Head Attention** : 여러 관점에서 토큰 간 관계를 학습
- **Feed Forward Network** : Attention 결과를 비선형 변환
- **Encoder** : 입력 문장의 의미와 문맥을 표현
- **Decoder** : 이전 출력 토큰을 참고하여 다음 토큰을 생성

### Transformer 모델 구조

| 구조 | 특징 | 대표 모델 | 주요 활용 |
| --- | --- | --- | --- |
| Encoder-only | 입력 문맥 이해에 집중 | BERT, KoELECTRA | 분류, 문장 이해, 임베딩 |
| Decoder-only | 다음 토큰을 반복적으로 예측 | GPT, Qwen, Llama | 텍스트 생성, LLM |
| Encoder-Decoder | 입력을 이해한 뒤 새로운 시퀀스 생성 | T5, NLLB | 번역, 요약, Seq2Seq |

### Decoder 기반 텍스트 생성

Decoder 기반 언어 모델은 이전에 등장한 토큰들을 이용하여 다음 토큰을 예측한다.

```text
입력 문장
    ↓
Tokenizer
    ↓
Transformer Decoder
    ↓
다음 토큰 확률 계산
    ↓
다음 토큰 선택
    ↓
생성된 토큰을 입력에 추가
    ↓
반복
    ↓
문장 생성
```